# Memory Buffer for Streaming Video-LLM Systems

**Assignment** — Hierarchical memory buffer that watches a video
stream, stores useful visual context under a fixed budget, retrieves relevant
past evidence for a text query, and formats that evidence for downstream reasoning.


Let's start by discussing my approach and architecture in general.

### Approach

I would want to start with outlining the design options that were possible:
- flat recent-frame buffer, which is simple but forgets older context (although proven to be quite a strong baseline on current streaming video benchmarks [[1]](https://arxiv.org/abs/2604.02317))
- text-only memory, which is easy to query but throws away visual detail
- full multimodal system, which is powerful and probably the best but too expensive under the limitations of this assignment 

Therefore, I chose a lightweight hierarchical visual memory, inspired by Fluxmem idea [[2]](https://arxiv.org/abs/2603.02096): recent windows are stored densely, older content is compressed into episodes and events, retrieval is coarse-to-fine (inspired by VideoTree [[4]](https://arxiv.org/abs/2405.19209)), and text summaries are used as an helper signal for retrieval and also additional information for LLM. This fits the assignment well because it keeps memory bounded, stays query-based, remains visual-first, and can be implemented cleanly in a notebook with small models.

### Architecture

The system follows a perception -> memory -> reasoning design [[5]](https://arxiv.org/abs/2412.09596). A video stream is first converted into short local windows, and each window is mapped to a compact visual embedding using a lightweight encoder. These embeddings are written into a hierarchical memory buffer with three levels [[2]](https://arxiv.org/abs/2603.02096): recent memory for dense short-term storage, episodic memory for merged coherent actions, and long-term event memory for compressed higher-level context.

When a text query arrives, it is encoded into the same retrieval space and used for coarse-to-fine retrieval [[4]](https://arxiv.org/abs/2405.19209): first over recent windows (always included), then over long-term events, then over episodic memory inside the selected regions, with final grounding from episode member windows. Text summaries are included as a supporting signal for retrieval, while the main memory representation remains visual.

The retrieved evidence is then formatted into a readable block that shows how it would be passed to an LLM for downstream reasoning. This final step was the most time-consuming part for me, because I spent a lot of time thinking about the best way to connect retrieved visual information to an LLM. In modern VLMs, this is usually done with a learned bridge such as an MLP projector that maps vision features into the language model space. Since I do not have access to such a pretrained bridge, and training one is outside the scope of this assignment, I had to simplify this part of the pipeline.

My first idea was to use lightweight visual embeddings only for retrieval, and then pass the retrieved frames to a stronger VLM for reasoning. That would still be a reasonable visual-first design. However, the assignment explicitly asks to show how retrieved information would be passed to an LLM. Under that constraint, my final choice was to use visual embeddings for retrieval and use episode/event summaries as the LLM-facing input. This is a simplification, but I think it is the most practical and honest solution here.

Detailed implementation choices (like model selection and hyperparameters, etc.) are described later in the notebook.

Here is my architecture sketch:

![Architecture](architecture.svg)

---
## 0. Setup

In [2]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").is_dir():
    pass
elif (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
else:
    ROOT = Path("..").resolve()

sys.path.insert(0, str(ROOT))

---
## 1. Stream Reader

Let's start by looking at my stream reader design. It is fairly simple: it opens a video file with OpenCV, subsamples frames at roughly fps, groups those sampled frames into consecutive chunks of about `fps * window_duration frames`, and yields each chunk as a RawWindow with `start_time`, `end_time`, a random-looking `window_id`, and RGB frame tensors. The last chunk is emitted even if it’s short. `RawWindow.representative_frame` is the middle frame of that list.

Implemented in `stream_reader.py`.

In [4]:
from src import StreamReader, RawWindow

---
## 2. Perception Encoder

Now, let's move on to the first essential part, the perception encoder. For this task, I decided to use X-CLIP (microsoft/xclip-base-patch32) because it meets the conditions I wanted:
- it runs frame features through a temporal fusion block (MIT), so each window vector encodes multi-frame context over a few seconds
- puts video and text in a shared embedding space for retrieval
- is still light enough for a notebook-style pipeline

**Video side.** For each window, encode_frames / encode_window takes the window’s RGB frames, uniformly subsamples up to 8, runs the X-CLIP vision stack, and passes frame features through the MIT temporal module. The result is one L2-normalised clip vector per window.

**Text side.** Here is a mild disadvantage. The tokenizer’s maximum length is small (77 subword tokens), so long summaries cannot be encoded in one shot. I split the token id sequence with `_chunk_token_ids`: fixed-size chunks, encode each chunk with `get_text_features`, then average the chunk embeddings and L2-normalise again so long text is still a single vector in the same space as queries and visuals.

Implemented in `perception_encoder.py`.

In [5]:
from src import PerceptionEncoder

## 3. Memory Buffer Design

This is probably the most important part of my project. My design is a three-tier hierarchical buffer that prioritizes a bounded context over keeping everything — inspired by HEM-style hierarchical episodic memory [[1]](https://arxiv.org/abs/2502.06975) and FluxMem's redundancy filtering [[2]](https://arxiv.org/abs/2603.02096).

<div align="center">
  <img src="MemoryBuffer.svg" width="75%" />
</div>

### What I store

Let's firstly look at `data_structures.py`. It defines the objects the rest of the pipeline passes around:

- `WindowEntry` — one short time segment: L2-normalised visual embedding, optional caption + text embedding, optional RGB frame for summarisers, and a tier flag (`"recent"` vs `"episodic"`). The helper `from_raw_window` builds it from a `RawWindow` plus an embedding. This is my *"right now"* memory.

- `EpisodeEntry` — a merged action span: pooled visual vector over member windows, `member_window_ids`, `representative_window_ids` (for captions / VLM), and episode-level summary text (+ summary embedding). This is my *"this activity"* memory.

- `EventEntry` — long-term chunk: centroid-style visual vector over episodes, `member_episode_ids`, `representative_window_ids`, and event-level summary. This is my *"the whole story"* memory.

- `RetrievalResult` — bundles coarse event hits, fine episode hits, grounded windows, and scores; we will use it again in the retrieval / formatter section.

Exactly how I build per-window captions and episode / event summaries will be explained later. Three tiers give me three levels of detail at query time: fine grounding in the recent queue, action-level semantics in episodes, story-level semantics in events. I need all three because a single query can mean any of them.

Now, let me describe my memory buffer design in depth.

### How memory size is controlled

Each tier has its own capacity rule:

- **Recent** — hard FIFO cap: `recent_capacity` ($20$ by default). When it overflows, the oldest window is popped and sent to the promotion pipeline.
- **Episodic** — soft cap: `episodic_capacity` ($50$ by default). Crossing it triggers `_consolidate_episodic()`, which greedily grows a contiguous cluster of up to `episodic_merge_batch` episodes, as long as the gap to the running group stays under `event_max_gap` and the cosine similarity to the running centroid stays above `event_min_episode_sim`, and then it rolls them into one `EventEntry`.
- **Long-term** — no hard cap. Events grow much slower than windows arrive (roughly 1 per 100+ windows), so letting this tier grow is cheap. It's also the densest-per-byte tier, so it's the last thing I'd want to drop.

**Write path.** Each new window lands in the recent queue. First thing, if it has a `summary_text` but no embedding yet, `HierarchicalMemoryWriter.update` encodes the caption on the spot, so the retriever's text channel is live from the very first tier. When the queue overflows, the oldest window gets a novelty score,

$$
\text{novelty} = 1 - \max \text{cosine\_similarity}
$$

against its remaining neighbours. Below `novelty_threshold` it's dropped; above it, it moves into the pending episode. This way the buffer keeps only moments that actually add something instead of filling up with near-duplicate frames — FluxMem's filter-temporal-redundancy idea [[2]](https://arxiv.org/abs/2603.02096).

The pending episode flushes on any natural boundary:

- a gap larger than `episode_max_gap`, or
- a similarity drop below `episode_min_sim`, or
- length hitting `episode_max_len`.

On flush, time-aware self-centrality pooling produces the episode's visual vector, and the top `n_rep_windows` members (by weight) become the representative frames for captioning. Captions come from a pluggable `summary_fn` (i chose Moondream2).

When the episodic tier goes over its cap, the consolidated cluster becomes an `EventEntry` with a norm-weighted centroid over its episodes, inherited representative windows, and a summary from a VLM fuser (`Qwen2.5-VL-3B-Instruct`) over sampled episode frames plus the concatenated episode texts.

At end of stream, `finalize()` drains the recent queue through the same novelty path (so the tail isn't stranded as `tier="recent"`), flushes the pending episode, and keeps consolidating until the episodic tier stops shrinking.


### What stays and what gets removed

Two independent policies decide fate:

**1. Novelty gate at Recent → Episode.** When a window gets popped from the Recent queue, I score its novelty as $1 - \max \text{cosine\_similarity}$ against its remaining Recent neighbours. Below `novelty_threshold` it's dropped entirely; above, it enters the pending episode. The reason is simple: a lot of videos have long stretches where adjacent frames look identical. Thus, we need this gate.

**2. Natural-boundary flush at the Episode level.** On flush, I compute a time-aware self-centrality weight for each member.

For formality, here are the exact formulas. Take an episode with $n$ members at timestamps $t_1 \le \dots \le t_n$ and L2-normalised embeddings $\mathbf{e}_1, \dots, \mathbf{e}_n$. Two per-member scores:

- a temporal-centrality term, a Gaussian pulling members toward the episode midpoint $t_{\text{mid}} = (t_1 + t_n)/2$,

$$
c_i = -\frac{(t_i - t_{\text{mid}})^2}{2\,\sigma_{\text{center}}^2};
$$

- a local-consistency term, a time-decayed sum of cosine similarities to the other members,

$$
s_i = \sum_{j \neq i} \exp\!\left(-\frac{|t_i - t_j|}{\sigma_{\text{time}}}\right) \cdot \cos(\mathbf{e}_i, \mathbf{e}_j).
$$

They combine linearly and go through a softmax over members,

$$
\ell_i = \lambda_{\text{center}} \cdot c_i + \mu_{\text{consistency}} \cdot s_i,
\qquad
w_i = \frac{\exp(\ell_i)}{\sum_{k=1}^{n} \exp(\ell_k)},
$$

and the episode's visual embedding is the L2-normalised weighted average:

$$
\mathbf{e}_{\text{episode}} = \frac{\sum_{i=1}^{n} w_i\, \mathbf{e}_i}{\left\lVert \sum_{i=1}^{n} w_i\, \mathbf{e}_i \right\rVert_2}.
$$

The top `n_rep_windows` members by $w_i$ become the representative frames fed to the VLM captioner. The rest of the members stay in the window archive so the retriever can still pull them as grounding context.

**3. Consolidation at the Event level.** When episodic capacity is exceeded, the oldest coherent cluster gets rolled into one event. The rolled-up episodes drop out of the episodic tier; their member windows stay in the window archive. Losing the episode is on purpose — the event summary keeps the meaning at lower resolution, and the representative frames are still reachable for grounding.


### How I build summaries at different levels

Each tier has its own captioner, picked for what that tier actually needs.

- **Window — Florence-2 `<CAPTION>`.** One short caption per window, greedy decode, `max_new_tokens=96`. It runs on every window at ingest, so it has to be cheap.

- **Episode — Moondream2, one caption per member frame.** On flush, I run Moondream2 on each window's representative frame and stitch the captions together with `[t=...s | frame .../...]` tags. The prompt tells it to stick to what's clearly visible and not to guess names, teams, brands, or on-screen text (hallucinations were an issue when I was testing on a football match). Going per-frame keeps the detail high without giving the model room to invent a story. Florence `<DETAILED_CAPTION>` is the fallback when Moondream is off.

- **Event — Qwen2.5-VL-3B-Instruct multi-image fusion.** On consolidation, I take the representative frames from every episode in the cluster, subsample down to `vlm_max_frames=10`, and hand them to Qwen together with the episode texts as a bulleted `Scene i [t_s–t_e]: ...` list. The prompt treats the images as the truth and the episode captions as hallucination-prone, so the model re-grounds any named entity against the pixels.

So the pattern is simple: windows get a one-liner because speed matters, episodes get per-frame grounded detail because we're already paying the flush cost, events get a long VLM fusion because that's the tier where text does most of the heavy lifting.


### Motivation behind the choices

Honestly, a lot of these calls came from testing things and seeing what broke. A few main ideas shaped the design:

- **Higher the tier, stronger the compression.** Text takes over from vision as we go deeper. At the window level I lean on vision: the raw embedding carries most of the signal and captions are short. At the episode level the two sit side by side. At the event level I flip it — a long VLM summary does the heavy lifting and vision is just a thin anchor. The deeper I go in the hierarchy, the more text replaces pixels. This is the core compression idea.

- **Why self-centrality pooling and not a plain mean.** Mean pooling treats every window in an episode as equally important, but they are not. Self-centrality gives more weight to windows that are both close to the episode midpoint *and* visually agree with their peers, so the pooled vector sits on the "typical" part of the episode instead of being dragged around by outliers. I initially considered better options like a recurrent memory update or full attention pooling, but I could not find open, cheap models for that at streaming rates. So I went with this light, attention-style version: softmax weights over hand-crafted scores, no extra parameters to train.

- **Model choices — what I tried.** At the window tier I tested BLIP and Florence-2; Florence-2 with `<CAPTION>` was faster and more grounded, so it stays. At the episode tier Florence-2 started inventing team names and player identities (when I took football match as a sample video), so I swapped to Moondream2 with an explicit anti-hallucination prompt — much cleaner on representative frames. For event fusion I first tried `Qwen2-VL-2B`; it was fluent but could not keep track of scores across multiple episodes. `Qwen2.5-VL-3B-Instruct` was the smallest one that held multi-episode state together, so that is what the event tier uses.

- **Write-path is query-agnostic.** The buffer is built without knowing what will be asked. The retriever is where query-conditioned ranking happens, and it works better on a uniform memory than on a query-biased one. So if I ever swap the retriever (for example, for a learned re-ranker), the buffer does not need to change.


In [6]:
from pathlib import Path

from src import (
    HierarchicalMemoryWriter,
    PerceptionEncoder,
    StreamReader,
    SummaryBuilder,
    WindowEntry,
)
from src.data_structures import EpisodeEntry, EventEntry, RetrievalResult

Let's use a short StreamingBench sample video to walk through the full flow. 

To download the data, run `download_video_sample.py`.

In [7]:
from IPython.display import Video, display

ROOT = Path("..").resolve()
video_path = ROOT / "data/streamingbench/Real_Time_Visual_Understanding/shard_1_50/sample_1/video.mp4"

display(Video(str(video_path), width=640, html_attributes="controls"))

First, some helper functions:

In [8]:
import textwrap
import time

W = 68
RC = 20
EC = 10
EG = 4.0
VG = 15.0
NT = 0.05


def rule(c="─"):
    print(c * W)


def wrap_box(s):
    w = W - 6
    out = []
    for x in s.splitlines() or [""]:
        out += textwrap.wrap(x, width=w) or [""]
    return out


def show_win(i, raw, txt):
    print(f"  [{raw.start_time:7.1f} – {raw.end_time:6.1f}s]  win {i:03d}  {txt}", flush=True)


def show_ep(ep, i):
    head = f"  episode {i:02d}  [{ep.start_time:.1f} – {ep.end_time:.1f}s]  {len(ep.member_window_ids)} windows"
    print(f"\n  ┌{'─' * (W - 4)}┐")
    print(f"  │  {head:<{W - 6}}  │")
    for line in wrap_box(ep.summary_text):
        print(f"  │  {line:<{W - 6}}  │")
    print(f"  └{'─' * (W - 4)}┘")


def show_ev(ev, i):
    head = f"  EVENT {i:02d}  [{ev.start_time:.1f} – {ev.end_time:.1f}s]  {len(ev.member_episode_ids)} episodes"
    print(f"\n  ╔{'═' * (W - 4)}╗")
    print(f"  ║  {head:<{W - 6}}  ║")
    for line in wrap_box(ev.summary_text):
        print(f"  ║  {line:<{W - 6}}  ║")
    print(f"  ╚{'═' * (W - 4)}╝")


def flush(mem, a, b):
    while a < len(mem.episodic):
        a += 1
        show_ep(mem.episodic[a - 1], a)
    while b < len(mem.long_term):
        b += 1
        show_ev(mem.long_term[b - 1], b)
    return a, b

Now the essential part:

In [9]:
import os
import logging

os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)

In [10]:
root = Path.cwd()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent

video = root / "data/streamingbench/Real_Time_Visual_Understanding/shard_1_50/sample_1/video.mp4"
if not video.is_file():
    raise FileNotFoundError(f"not found: {video}")

print("Loading summary stack...")
summary = SummaryBuilder(use_model=True, use_vlm=True, use_moondream=True)

enc = PerceptionEncoder()
reader = StreamReader(fps=1.0, window_duration=3.0)

mem = HierarchicalMemoryWriter(
    recent_capacity=RC,
    episodic_capacity=EC,
    novelty_threshold=NT,
    episode_max_gap=EG,
    event_max_gap=VG,
    summary_fn=summary,
    text_encode_fn=enc.encode_text,
    store=None,
)

lim = None

rule("━")
print(f" Streaming: {video.name}")
print(" Summaries: Florence-2 · Moondream2 · Qwen2.5-VL")
rule("━")
print()

t0 = time.perf_counter()
n = a = b = 0

for i, raw in enumerate(reader.read_windows(str(video))):
    if lim is not None and i >= lim:
        break
    vis = enc.encode_window(raw)
    txt = summary.build_window_caption(raw)
    mem.update(WindowEntry.from_raw_window(raw, visual_embedding=vis, summary_text=txt))
    n += 1
    show_win(n, raw, txt)
    a, b = flush(mem, a, b)

mem.finalize()
a, b = flush(mem, a, b)

dt = time.perf_counter() - t0
s = mem.stats()

print()
rule("━")
print(f" {n} windows  ·  {s['n_episodes_flushed']} episodes  ·  {s['long_term']} events  ·  {dt:.1f}s")
print(f" promoted {s['n_promoted']}  ·  discarded {s['n_discarded']}  ·  recent {s['recent']}  ·  episodic {s['episodic']}")
rule("━")

Loading summary stack...


Loading captioner microsoft/Florence-2-base on cuda (torch.float16)…
Captioner ready.
Loading event VLM Qwen/Qwen2.5-VL-3B-Instruct on cuda (torch.bfloat16)…


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Event VLM ready.
Loading Moondream vikhyatk/moondream2 on cuda (torch.bfloat16)…
Moondream ready.
Loading microsoft/xclip-base-patch32 on cuda...
Ready. Embedding dim = 512.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Streaming: video.mp4
 Summaries: Florence-2 · Moondream2 · Qwen2.5-VL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  [    0.0 –    2.0s]  win 001  a red and blue background with the word subscribe on it
  [    2.0 –    5.0s]  win 002  A man in a black shirt standing in front of a crowd.
  [    5.0 –    8.0s]  win 003  A couple of men standing next to each other on a field.
  [    8.0 –   11.0s]  win 004  a soccer field with a circle in the middle of it
  [   11.0 –   14.0s]  win 005  a soccer game is being played in a stadium
  [   14.0 –   17.0s]  win 006  a soccer game is being played in a stadium
  [   17.0 –   20.0s]  win 007  A group of people playing soccer on a field.
  [   20.0 –   23.0s]  win 008  A group of peop

*P.S. You may need to view as scrollable element to see all the outputs.*

By the way, the match was indeed between Sevilia FC and Barcelona FC.

## 4. Retrieval

The next vital part of this system is the retrieval. My retriever follows one-to-one, coarse-to-fine idea, inspired by VideoTree [[4]](https://arxiv.org/abs/2405.19209).

Implemented in `retriever.py` as `HierarchicalRetriever.retrieve(...)`.

### Query encoding

I encode `Q` with the same X-CLIP text tower I already use for summaries (`PerceptionEncoder.encode_text`), so the query lands in the same L2-normalised 512-dim space as windows, episodes, events, and summary embeddings. That shared space is the whole reason X-CLIP was picked as the retrieval encoder — cosine similarity between text and visuals is meaningful, so no projector needed.

Long queries are chunked via `_chunk_token_ids`, encoded per chunk and averaged, so the retriever always sees one 512-dim vector regardless of query length.

### Three-stage coarse-to-fine search

Once I have `q_emb`:

**Stage 0 — always search recent windows.** Cosine-score the recent deque against `q_emb`, keep top-K. This is my SimpleStream [[1]](https://arxiv.org/abs/2604.02317) safety net: recent context is a strong baseline, so the retriever must always include fresh evidence. I think that for "what is happening right now?" queries this stage carries most of the signal.

**Stage A — coarse routing over events.** Score every `EventEntry`, apply the temporal prior (below), keep top-M (default is $3$). Output: candidate time ranges. Cheapest stage — events are the tightest tier, typically tens of them.

**Stage B — fine search over episodes.** Take episodes overlapping any stage-A range, **union** with the last `recent_episodes` trailing ones (including the still-forming `_pending_episode` ), score, keep top-K. Also, the end of the stream is like the very start—stuff isn’t in an event yet. A coarse-only path would have quietly skipped it. The tail bypass matches stage 0 so that doesn’t happen; I actually saw that break in evals before the fix.

**Stage C — grounding.** For each top episode, pull back its member windows from `_window_archive` via `get_grounding_windows`. Stage-0 hits and stage-C windows are deduped by `entry_id` into one `grounded_windows` list.

So one call returns three parallel views — **coarse** (events), **fine** (episodes), **grounded** (windows) — bundled into a `RetrievalResult` with a score map for provenance.

### Scoring — from additive α/β/γ to multiplicative decay

I iterated on this part the most — and it’s worth telling, because the first version looked fine and still broke.

I started with the obvious blend — visual, text, recency, all additive:

$$
\text{score}(e, q) = \alpha \cdot \cos(e_\text{vis}, q_\text{vis}) + \beta \cdot \cos(e_\text{sum}, q_\text{txt}) + \gamma \cdot \text{recency\_bonus}(e),
$$

with $\alpha + \beta + \gamma = 1$, visual dominant, recency_bonus a linear falloff.

On a five-QA cooking video, the intro event $[0\text{–}68\text{s}]$ won a coarse-top-3 slot for all five queries — regardless of what was asked. It is the longest event in the stream (most tokens, most surface area for keyword overlap with any "cooking / kitchen" query) and its visual centroid is basically "generic kitchen", which matches every cooking query visually too.

So I turned $\gamma$ up. $0.05 \to 0.15 \to 0.25$. Intro event still won.

Turns out $\gamma$ only bumps the near candidate; it doesn’t hit the far one. The intro was so far ahead on meaning that a bigger $\gamma$ couldn’t catch up — I was optimising the thing that can’t fix that problem.

Fix: make the temporal term **multiplicative**:

$$
\boxed{\text{score}(e, q) = \bigl(\alpha \cdot \cos(e_\text{vis}, q_\text{vis}) + \beta \cdot \cos(e_\text{sum}, q_\text{txt})\bigr) \cdot \exp\!\bigl(-\text{dt}(e, q_\text{time}) / \tau\bigr)}
$$

with $\alpha = 0.70$, $\beta = 0.30$, $\tau = 0.5 \cdot \text{stream\_span}$ (so half a span away scales to $\approx 0.37$, full span to $\approx 0.14$), and $\text{dt} = 0$ if the entry straddles `query_time`. $\gamma$ disappears as a parameter.

Worth mentioning, two later tweaks sharpened this. $\text{stream\_span}$ is a **unified** span across recent $\cup$ episodes $\cup$ events — one ruler for all tiers, so an event straddling `query_time` now also gets $\text{decay} \approx 1$ instead of being penalised by a tier-local span. The visual term then splits by tier: for events it is $$\max_{w \in \text{reps}(e)} \cos(w_\text{vis}, q_\text{vis})$$ — peak over representative windows — because event embeddings are centroids-of-centroids and that 2-stage averaging blurs the semantic match; for episodes we keep the pooled centroid, since self-centrality pooling already weights members by centrality and consistency, so the episode vector sits on the "typical" frame and doesn't suffer the same blur. Windows use their own embedding directly. The centroid still drives write-time decisions for both tiers (novelty gate, event clustering).

Multiplicative decay scales the semantic score down by how far the entry is — so a distant entry has to genuinely earn its place. Same basic idea as ‘fresh’ ranking in feeds and decayed BM25. After the switch, the intro stopped winning later-content queries.

When `query_time=None`, the decay collapses to $1.0$ and ranking degrades to plain semantic cosine.

### What `retrieve()` returns — the LLM context subset

```python
RetrievalResult(
    query=q,                              # raw text, for provenance
    coarse_hits=[EventEntry, ...],        # top-M events — compressed long-term context
    episodic_hits=[EpisodeEntry, ...],    # top-K episodes — concrete actions
    grounded_windows=[WindowEntry, ...],  # stage 0 ∪ stage C, deduped — fine evidence
    scores={entry_id: float},
)
```

The formatter (next section) turns this into an LLM-facing block — coarse hits are the "story so far", episodes are the "relevant chapters", grounded windows are the "actual frames to cite". Returning three levels instead of one flat top-K is the whole point of hierarchical memory; a flat list would waste it.

### Why this suits streaming video understanding

- **Bounded read cost.** Stage A is $O(\lvert\text{events}\rvert)$, B is $O(\lvert\text{candidate episodes}\rvert)$, C is $O(\lvert\text{members of top-K}\rvert)$. None grow as fast as raw windows — events compress more aggressively over time. Therefore, retrieval stays cheap at hour-long inputs.
- **Query-agnostic write, query-aware read.** The buffer doesn't know future questions; the retriever is fully query-conditioned. I can swap in a learned re-ranker later without touching storage — the same perception / memory / reasoning split IXC2.5-OmniLive [[5]](https://arxiv.org/abs/2412.09596) argues for.
- **Recency is structural.** Stage 0 always runs, the tail bypass always runs, and multiplicative decay biases ranking toward the present without throwing old content away.
- **Visual-first, text-assisted.** $\alpha=0.70$ keeps visual dominant; $\beta=0.30$ lets captions help queries where the pixel match is weak but the text is strong. Summaries are an index, not the memory.


Here’s the retriever in action on the following video.

In [11]:
from IPython.display import Video, display

ROOT = Path("..").resolve()
video_path = ROOT / "data/streamingbench/Real_Time_Visual_Understanding/shard_1_50/sample_36/video.mp4"

display(Video(str(video_path), width=640, html_attributes="controls"))

Some helper functions:

In [12]:
import json
import re
import textwrap

from src import HierarchicalRetriever, ReasonerInputFormatter

WRAP_WIDTH = 100


def wrap_formatted(text):
    out = []
    for line in text.splitlines():
        if len(line) <= WRAP_WIDTH:
            out.append(line)
            continue
        indent = re.match(r"\s*", line).group(0)
        hang = indent + "    "
        out.append(
            textwrap.fill(
                line,
                width=WRAP_WIDTH,
                initial_indent="",
                subsequent_indent=hang,
                break_long_words=False,
                break_on_hyphens=False,
            )
        )
    return "\n".join(out)


def hms_to_seconds(ts):
    parts = [float(p) for p in ts.strip().split(":")]
    if len(parts) == 3:
        h, m, s = parts
        return h * 3600 + m * 60 + s
    if len(parts) == 2:
        m, s = parts
        return m * 60 + s
    return parts[0]


def show_qa(qa, stream_time):
    idx = qa["question_id"].split("_")[-1]
    print()
    rule("═")
    print(f"  QA {idx}  ·  fired at stream t = {stream_time:.1f}s  "
          f"(qa timestamp = {qa['t_seconds']:.1f}s)")
    print(f"  task: {qa['task_type']}")
    print(f"  Q:    {qa['question']}")
    for opt in qa["options"]:
        print(f"        {opt}")
    print(f"  ground truth: {qa['answer']}")
    rule("═")


def fire_due(cursor, stream_time, qas, mem, retriever, formatter):
    if cursor >= len(qas) or qas[cursor]["t_seconds"] > stream_time:
        return cursor

    mem.flush_pending()

    while cursor < len(qas) and qas[cursor]["t_seconds"] <= stream_time:
        qa = qas[cursor]
        q_emb = enc.encode_text(qa["question"])
        result = retriever.retrieve(
            query=qa["question"],
            query_embedding=q_emb,
            memory=mem,
            top_m=3,
            top_k=5,
            query_time=stream_time,
        )
        show_qa(qa, stream_time)
        print(wrap_formatted(formatter.format_text(result)))
        cursor += 1
    return cursor

In [13]:
import os
import logging

os.environ["TQDM_DISABLE"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
logging.getLogger("transformers").setLevel(logging.ERROR)

Now, the essential part:

In [14]:
video36 = root / "data/streamingbench/Real_Time_Visual_Understanding/shard_1_50/sample_36/video.mp4"
qas_path = video36.parent / "qas.json"

qas = json.loads(qas_path.read_text())["qas"]
for qa in qas:
    qa["t_seconds"] = hms_to_seconds(qa["time_stamp"])
qas.sort(key=lambda q: q["t_seconds"])

mem = HierarchicalMemoryWriter(
    recent_capacity=RC,
    episodic_capacity=EC,
    novelty_threshold=NT,
    episode_max_gap=EG,
    event_max_gap=VG,
    summary_fn=summary,
    text_encode_fn=enc.encode_text,
    store=None,
)
retriever = HierarchicalRetriever()
formatter = ReasonerInputFormatter()

cursor = 0

for raw in reader.read_windows(str(video36)):
    vis = enc.encode_window(raw)
    txt = summary.build_window_caption(raw)
    mem.update(WindowEntry.from_raw_window(raw, visual_embedding=vis, summary_text=txt))
    cursor = fire_due(cursor, raw.end_time, qas, mem, retriever, formatter)

mem.finalize()
cursor = fire_due(cursor, float("inf"), qas, mem, retriever, formatter)



════════════════════════════════════════════════════════════════════
  QA 1  ·  fired at stream t = 203.0s  (qa timestamp = 202.0s)
  task: Attribute Perception
  Q:    What color are the kitchen cupboards?
        A. Dark blue.
        B. Light green.
        C. White.
        D. Yellow.
  ground truth: B
════════════════════════════════════════════════════════════════════
QUERY: What color are the kitchen cupboards?

[Coarse — event-level routing hits]
  [0.0–80.0s] sim=0.113  Event 0.0–80.0s: The video begins by introducing Gordon Ramsay, a renowned
      chef, set against a simple yet striking backdrop featuring bold red letters spelling out
      "RAMSAY," accompanied by smaller text reading "in 10." This opening sets the stage for a
      culinary journey led by Ramsay himself. As the video progresses, we see Ramsay actively
      engaged in cooking demonstrations within a modern, well-equipped kitchen. His presence is
      commanding, often seen standing confidently next to st

## 5. LLM input integration

OK so the last piece of the system is: retrieval gives me three parallel views — coarse events, fine episodes, grounded windows — how do these actually become the thing an LLM reads?

Implemented in `formatter.py` as `ReasonerInputFormatter`.

### What would go into a real VLM

In a real multimodal pipeline there are basically three ways to feed retrieved visual memory into a language model:

- **Projector / vision-LM bridge.** Train a small MLP that maps each retrieved visual embedding into the LLM's token space, concatenate those projected tokens with the text query, and let the LLM attend over them directly. This is what LLaVA-style models do, and this is cheap at inference, but requires a trained projector, which is out of scope for this assignment. 
- **Feed the raw frames into a VLM.** Skip the embedding layer entirely — pull the representative frames back from `_window_archive`, pass them to a VLM like Qwen2.5-VL alongside the query, and let the VLM's own vision tower do the encoding. More expensive per call, but no custom training and the VLM's visual understanding is strictly stronger than a frozen X-CLIP feature.
- **Q-Former.** Learn a small set of query tokens that cross-attend into the visual embedding set and distil it down to a short sequence of "answer-oriented" tokens — this is BLIP-2's trick [[6]](https://arxiv.org/abs/2301.12597).

These are the architectural solutions I considered the strongest. But this is a lightweight notebook demo — I don't have a trained projector, I'm not going to fine-tune a Q-Former in one week, and running Qwen-VL on every QA is too slow for an interactive loop. So I do the thing that's honest for the scope: I pass the retrieved text information — summaries, window captions, timestamps, sim scores — straight into the LLM context, and leave the visual-embedding slot wired up but unused in this demo.

### What I actually pass to the LLM

`ReasonerInputFormatter.format_for_llm(result, query_embedding)` returns a dict that's explicit about both the "real" and "simplified" paths:

```python
{
  "query":             str,                       # raw text
  "query_embedding":   List[float] | None,        # 512-dim X-CLIP text vec
  "visual_context":    [                          # one entry per retrieved item
    {
      "source":         "episodic" | "grounding_window",
      "time_range":     [start_s, end_s],
      "embedding_dim":  512,
      "embedding":      List[float],              # L2-normalised X-CLIP vis vec
      "summary":        str,                      # Moondream / Florence caption
      "score":          float,                    # retrieval sim
    },
    ...
  ],
  "text_context":          str,                   # human-readable evidence block
  "n_visual_tokens":       int,
  "coarse_event_count":    int,
  "episodic_hit_count":    int,
}
```

Two channels on purpose:

- **`visual_context`**: every retrieved episode and grounding window ships its raw 512-dim visual embedding plus its time range, summary, and sim score. This is what a projector / Q-Former / VLM bridge would consume — the data is already in the dict.
- **`text_context`** is the simplified path I actually use. It's the same string `format_text(result)` produces: the coarse event paragraphs, the episodic summaries, and the grounded window captions — each tagged with its time range and similarity score. This is the *readable proxy* for the visual tokens. An LLM reading this string gets the "story so far" (coarse), the "relevant chapters" (episodes), and the "actual frames to cite" (grounding), all with timestamps it can reference in its answer.

### How the query and visual context combine

For the simplified demo, the LLM prompt is basically:

```
[system] You are answering a question about a video.
         Use the evidence below. Every line has a time range.
         Cite the time ranges you used.

[evidence]
<text_context>

[user]
Query: <query>
```

Again, in the full-system interpretation, the evidence block would be replaced by (or concatenated with) the projected visual tokens from `visual_context`, and the query would share the same attention window. The shape of the prompt doesn't really change; what changes is that the "evidence" section becomes a mix of projected-embedding tokens and text tokens instead of only text.

Let's look at this component in action on the same sample video.

In [18]:
video36 = root / "data/streamingbench/Real_Time_Visual_Understanding/shard_1_50/sample_36/video.mp4"
qas = json.loads((video36.parent / "qas.json").read_text())["qas"]
for qa in qas:
    qa["t_seconds"] = hms_to_seconds(qa["time_stamp"])
qas.sort(key=lambda q: q["t_seconds"])

mem = HierarchicalMemoryWriter(
    recent_capacity=RC, episodic_capacity=EC, novelty_threshold=NT,
    episode_max_gap=EG, event_max_gap=VG, summary_fn=summary,
    text_encode_fn=enc.encode_text, store=None,
)
retriever, formatter = HierarchicalRetriever(), ReasonerInputFormatter()


def show_llm_input(qa, stream_time, mem, retriever, formatter):
    mem.flush_pending()
    q_emb = enc.encode_text(qa["question"])
    result = retriever.retrieve(
        query=qa["question"], query_embedding=q_emb, memory=mem,
        top_m=3, top_k=5, query_time=stream_time,
    )
    llm_input = formatter.format_for_llm(result, query_embedding=q_emb)

    show_qa(qa, stream_time)
    print(
        f"query embedding size : {len(llm_input['query_embedding'])}\n"
        f"visual tokens        : {llm_input['n_visual_tokens']}\n"
        f"coarse events found  : {llm_input['coarse_event_count']}\n"
        f"episodic matches     : {llm_input['episodic_hit_count']}\n"
        "\nVisual-context slots:\n"
        + "".join(
            "  [{i}] {src:<17} [{a:.1f}-{b:.1f}s]  dim={dim}  score={sc}\n".format(
                i=i, src=t["source"], a=t["time_range"][0], b=t["time_range"][1],
                dim=t["embedding_dim"],
                sc=f"{t['score']:.3f}" if t["score"] is not None else "  —  ",
            )
            for i, t in enumerate(llm_input["visual_context"])
        )
    )
    print(
        "\nExact text fed to the LLM:\n\n"
        "[system] You are answering a question about a video.\n"
        "Use the evidence below. Each line includes a time range.\n"
        "When you answer, mention the time ranges you relied on.\n\n"
        "[evidence]\n"
        f"{wrap_formatted(llm_input['text_context'])}\n\n"
        "[user]\n"
        f"Question: {llm_input['query']}"
    )


def fire_due(cursor, stream_time, qas, mem, retriever, formatter):
    while cursor < len(qas) and qas[cursor]["t_seconds"] <= stream_time:
        show_llm_input(qas[cursor], stream_time, mem, retriever, formatter)
        cursor += 1
    return cursor


cursor = 0
for raw in reader.read_windows(str(video36)):
    mem.update(
        WindowEntry.from_raw_window(
            raw, visual_embedding=enc.encode_window(raw), summary_text=summary.build_window_caption(raw)
        )
    )
    cursor = fire_due(cursor, raw.end_time, qas, mem, retriever, formatter)

mem.finalize()
fire_due(cursor, float("inf"), qas, mem, retriever, formatter)


════════════════════════════════════════════════════════════════════
  QA 1  ·  fired at stream t = 203.0s  (qa timestamp = 202.0s)
  task: Attribute Perception
  Q:    What color are the kitchen cupboards?
        A. Dark blue.
        B. Light green.
        C. White.
        D. Yellow.
  ground truth: B
════════════════════════════════════════════════════════════════════
query embedding size : 512
visual tokens        : 19
coarse events found  : 1
episodic matches     : 5

Visual-context slots:
  [0] episodic          [119.0-143.0s]  dim=512  score=0.196
  [1] episodic          [116.0-119.0s]  dim=512  score=0.170
  [2] episodic          [101.0-107.0s]  dim=512  score=0.124
  [3] episodic          [107.0-110.0s]  dim=512  score=0.115
  [4] episodic          [110.0-116.0s]  dim=512  score=0.112
  [5] grounding_window  [191.0-194.0s]  dim=512  score=0.303
  [6] grounding_window  [200.0-203.0s]  dim=512  score=0.300
  [7] grounding_window  [188.0-191.0s]  dim=512  score=0.297
  [8] gr

5

## 6. LLM Reasoner

This section actually feeds the evidence to a language model and gets a prediction back.

I picked `Qwen2.5-3B-Instruct` (text-only) because it’s the same `Qwen2.5` family as the event VLM, so we don’t juggle a second tokenizer stack; it’s small enough to run with Florence + Moondream + Qwen-VL in one session without thrashing; and text-only matches this path: the formatter still exposes `visual_context`, but this demo only uses text_context, so we’re not claiming VLM use where we don’t. Anything else (projector, frame-VLM, Q-Former) would be a drop-in swap for the reasoner and leave retrieval unchanged.

In [24]:
from src.llm_reasoner import LLMReasoner

video36 = root / "data/streamingbench/Real_Time_Visual_Understanding/shard_1_50/sample_36/video.mp4"
qas = json.loads((video36.parent / "qas.json").read_text())["qas"]
for qa in qas:
    qa["t_seconds"] = hms_to_seconds(qa["time_stamp"])
qas.sort(key=lambda q: q["t_seconds"])

mem = HierarchicalMemoryWriter(
    recent_capacity=RC, episodic_capacity=EC, novelty_threshold=NT,
    episode_max_gap=EG, event_max_gap=VG, summary_fn=summary,
    text_encode_fn=enc.encode_text, store=None,
)
retriever, formatter = HierarchicalRetriever(), ReasonerInputFormatter()
reasoner = LLMReasoner()

correct = 0


def answer_due(cursor, stream_time, qas, mem, retriever, formatter, reasoner):
    global correct
    while cursor < len(qas) and qas[cursor]["t_seconds"] <= stream_time:
        qa = qas[cursor]
        mem.flush_pending()
        q_emb = enc.encode_text(qa["question"])
        result = retriever.retrieve(
            query=qa["question"], query_embedding=q_emb, memory=mem,
            top_m=3, top_k=5, query_time=stream_time,
        )
        llm_input = formatter.format_for_llm(result, query_embedding=q_emb)
        prediction = reasoner.answer(llm_input, options=qa["options"])
        letter = next((c for c in prediction.strip() if c in "ABCD"), "?")
        correct += int(letter == qa["answer"])

        rule("─")
        print(f"  QA {cursor + 1}  ·  t={stream_time:.1f}s  ·  Q: {qa['question']}")
        print(f"  prediction  : {prediction}")
        print("  ground truth: " + next(
            (o for o in qa["options"] if o.startswith(qa["answer"] + ".")), qa["answer"]
        ))
        cursor += 1
    return cursor


cursor = 0
for raw in reader.read_windows(str(video36)):
    mem.update(
        WindowEntry.from_raw_window(
            raw, visual_embedding=enc.encode_window(raw), summary_text=summary.build_window_caption(raw)
        )
    )
    cursor = answer_due(cursor, raw.end_time, qas, mem, retriever, formatter, reasoner)

rule("═")
print(f"  MCQ accuracy : {correct}/{len(qas)}")
rule("═")

Loading reasoner LLM Qwen/Qwen2.5-3B-Instruct on cuda (torch.bfloat16)…


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Reasoner LLM ready.
────────────────────────────────────────────────────────────────────
  QA 1  ·  t=203.0s  ·  Q: What color are the kitchen cupboards?
  prediction  : C. White.
The kitchen cupboards are white based on the evidence from [128.0–131.0s], where a man is holding a frying pan in a kitchen setting, and the kitchen features white cabinets, marble countertops, and various kitchen appliances and utensils.
  ground truth: B. Light green.
────────────────────────────────────────────────────────────────────
  QA 2  ·  t=209.0s  ·  Q: What is the person doing with the box grater?
  prediction  : D. Grating a yellow ingredient.
[146.0–149.0s] sim=0.180  Episode 146.0–149.0s: A blonde man in a navy blue t-shirt is standing in a kitchen, actively grating something with a handheld grater.
  ground truth: D. Grating a yellow ingredient.
────────────────────────────────────────────────────────────────────
  QA 3  ·  t=404.0s  ·  Q: What is the person in a blue shirt chopping on the woo

In my opinion, this result is quite good given that we don't feed any vision embeddings into the LLM.

## 7. Baseline comparison

This section reruns exactly the same end-to-end loop, but with `RecentWindowBaseline` in place of `HierarchicalRetriever` — the SimpleStream-style [[1]](https://arxiv.org/abs/2604.02317) flat recent-frames buffer that most current streaming VQA systems use as their default, and a surprisingly strong one on short-horizon benchmarks.

Implemented in `baseline.py` as `RecentWindowBaseline`: a `deque(maxlen=N)` of `WindowEntry`, cosine-scored against `q_emb` at query time. No episodes, no events, no consolidation, no temporal decay — just the last N windows and nearest-neighbour search over them.

In [25]:
from src import RecentWindowBaseline, RetrievalResult
from src.memory_writer import cosine_sim

video36 = root / "data/streamingbench/Real_Time_Visual_Understanding/shard_1_50/sample_36/video.mp4"
qas = json.loads((video36.parent / "qas.json").read_text())["qas"]
for qa in qas:
    qa["t_seconds"] = hms_to_seconds(qa["time_stamp"])
qas.sort(key=lambda q: q["t_seconds"])

baseline = RecentWindowBaseline(n_windows=RC)
formatter = ReasonerInputFormatter()

correct_bl = 0


def answer_due_baseline(cursor, stream_time, qas, baseline, formatter, reasoner):
    global correct_bl
    while cursor < len(qas) and qas[cursor]["t_seconds"] <= stream_time:
        qa = qas[cursor]
        q_emb = enc.encode_text(qa["question"])
        hits = baseline.retrieve(q_emb, top_k=5)
        scores = {w.entry_id: float(cosine_sim(q_emb, w.visual_embedding)) for w in hits}
        result = RetrievalResult(
            query=qa["question"], coarse_hits=[], episodic_hits=[],
            grounded_windows=hits, scores=scores,
        )
        llm_input = formatter.format_for_llm(result, query_embedding=q_emb)
        prediction = reasoner.answer(llm_input, options=qa["options"])
        letter = next((c for c in prediction.strip() if c in "ABCD"), "?")
        correct_bl += int(letter == qa["answer"])

        rule("─")
        print(f"  QA {cursor + 1}  ·  t={stream_time:.1f}s  ·  Q: {qa['question']}")
        print(f"  prediction  : {prediction}")
        print("  ground truth: " + next(
            (o for o in qa["options"] if o.startswith(qa["answer"] + ".")), qa["answer"]
        ))
        cursor += 1
    return cursor


cursor = 0
for raw in reader.read_windows(str(video36)):
    baseline.update(
        WindowEntry.from_raw_window(
            raw, visual_embedding=enc.encode_window(raw), summary_text=summary.build_window_caption(raw)
        )
    )
    cursor = answer_due_baseline(cursor, raw.end_time, qas, baseline, formatter, reasoner)

rule("═")
print(f"  MCQ accuracy (baseline): {correct_bl}/{len(qas)}")
rule("═")


────────────────────────────────────────────────────────────────────
  QA 1  ·  t=203.0s  ·  Q: What color are the kitchen cupboards?
  prediction  : C. White.
[188.0–191.0s] sim=0.201  A man standing in a kitchen preparing food on a counter. This sentence suggests a white countertop, which is often associated with white kitchen cupboards.
  ground truth: B. Light green.
────────────────────────────────────────────────────────────────────
  QA 2  ·  t=209.0s  ·  Q: What is the person doing with the box grater?
  prediction  : D. Sim=0.240 A man grating an orange on a grater in a kitchen. The evidence directly mentions someone using a grater to grate an orange, which aligns with option D.
  ground truth: D. Grating a yellow ingredient.
────────────────────────────────────────────────────────────────────
  QA 3  ·  t=404.0s  ·  Q: What is the person in a blue shirt chopping on the wooden cutting board?
  prediction  : D
The person in a blue shirt is chopping onions as evidenced by the st

## 8. Actual Long Video

## References

[1] [A Simple Baseline for Streaming Video Understanding](https://arxiv.org/abs/2604.02317)

[2] [FluxMem: Adaptive Hierarchical Memory for Streaming Video Understanding](https://arxiv.org/abs/2603.02096)

[3] [FlexMem: Scaling the Long Video Understanding of Multimodal Large Language Models via Visual Memory Mechanism](https://arxiv.org/abs/2603.29252)

[4] [VideoTree: Adaptive Tree-based Video Representation for LLM Reasoning on Long Videos](https://arxiv.org/abs/2405.19209)

[5] [InternLM-XComposer2.5-OmniLive: A Comprehensive Multimodal System for Long-term Streaming Video and Audio Interactions](https://arxiv.org/abs/2412.09596)

[6] [StreamBridge: Turning Your Offline Video Large Language Model into a Proactive Streaming Assistant](https://arxiv.org/abs/2505.05467)